# Tournament Package — Module Walkthrough

A section-by-section explanation of the `tournament` simulation package with live examples for each file.

**Module dependency order (bottom-up):**
`models` → `generators` → `standings` → `pairing` → `engine` → `metrics` → `io`

See `../PLANNING.md` for the study design and `../docs/decisions.md` for design trade-offs.

In [1]:
import sys
from pathlib import Path

# Make the package importable from notebooks/
MODULE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(MODULE_ROOT))

from random import Random
import pandas as pd

print("Package loaded.")

Package loaded.


## 1. `models.py` — Core data model

Defines the fundamental objects used throughout the package.

- **`Player(pid, name, skill)`** — a competitor. `skill` is a latent true-strength value used only by generators; pairing algorithms never see it.
- **`Game(agents_a, agents_b)`** — outcome of one game. Won by the first player to capture `AGENTS_TO_WIN = 3` agents.
- **`Match(player_a, player_b, games, bonus_agents_a, bonus_agents_b)`** — exactly two games. Outcome is decided by *game wins* (2-0 = win, 1-1 = draw). Agent totals are surfaced for standings/tiebreakers only. `bonus_agents_*` breaks a tied agent score — set at construction by the generator, never mutated.
- **`Round(number, matches)`** — a list of matches plus a round number (1-indexed).
- **`Tournament(players, rounds)`** — the full event.

In [7]:
from tournament.models import AGENTS_TO_WIN, Player, Game, Match, Round, Tournament

print(f"Agents needed to win a game: {AGENTS_TO_WIN}")

p1 = Player(pid=1, name="Alice", skill=1.5)
p2 = Player(pid=2, name="Bob",   skill=0.5)
print(p1)
print(p2)

# Game where player A wins (captures 3), player B captures 2
g_a_wins = Game(agents_a=3, agents_b=2)
print("valid:", g_a_wins.is_valid, "| A wins:", g_a_wins.winner_is_a)

# Game where player B wins
g_b_wins = Game(agents_a=1, agents_b=3)
print("valid:", g_b_wins.is_valid, "| A wins:", g_b_wins.winner_is_a)

Agents needed to win a game: 3
Player(pid=1, name='Alice', skill=1.5)
Player(pid=2, name='Bob', skill=0.5)
valid: (True, '') | A wins: True
valid: (True, '') | A wins: False


In [9]:
# --- Match where player 1 wins both games (2-0) ---
m_win = Match(player_a=1, player_b=2, games=[Game(3, 1), Game(3, 2)])
print("=== 2-0 win ===")
print("winner:", m_win.winner)
print("is_draw:", m_win.is_draw)
print("total_agents A / B:", m_win.total_agents_a, "/", m_win.total_agents_b)
print("results[1]:", m_win.results[1])
print()

# --- Draw match (1-1 split). Agent totals are tied, so bonus_agents breaks the tie ---
# game 1: A wins (3-2), game 2: B wins (2-3) → agents without bonus: A=5, B=5
m_draw = Match(
    player_a=1, player_b=2,
    games=[Game(3, 2), Game(2, 3)],
    bonus_agents_a=1, bonus_agents_b=0,  # generator awards 1 bonus agent to A
)
print("=== 1-1 draw ===")
print("winner:", m_draw.winner)           # None — no game-win majority
print("is_draw:", m_draw.is_draw)
print("agent_score:", m_draw.agent_score) # (6, 5) — bonus breaks the standings tie
print("results[2]:", m_draw.results[2])

=== 2-0 win ===
winner: 1
is_draw: False
total_agents A / B: 6 / 3
results[1]: {'id': 1, 'wins': 2, 'losses': 0, 'player_agents': 6, 'opponent_agents': 3, 'bonus_agents': 0, 'opponent': 2}

=== 1-1 draw ===
winner: None
is_draw: True
agent_score: (6, 5)
results[2]: {'id': 2, 'wins': 1, 'losses': 1, 'player_agents': 5, 'opponent_agents': 6, 'bonus_agents': 0, 'opponent': 1}


In [10]:
p3 = Player(pid=3, name="Carol", skill=0.8)
p4 = Player(pid=4, name="Dan",   skill=1.2)
m2 = Match(player_a=3, player_b=4, games=[Game(2, 3), Game(3, 1)])

rnd = Round(number=1, matches=[m_win, m2])
print("Opponents this round:", rnd.opponents())

tour = Tournament(players=[p1, p2, p3, p4])
tour.rounds.append(rnd)
print("Past opponents of player 1:", tour.past_opponents(1))
print("Rounds played:", len(tour.rounds))

Opponents this round: {1: 2, 3: 4}
Past opponents of player 1: {2}
Rounds played: 1


## 2. `generators.py` — Result generators

Produces `Match` objects from two `Player` objects and an `rng`. These are the only functions that ever look at `Player.skill`.

- **`make_players(n, rng, skill_sd=1.0)`** — creates `n` players with normally distributed skills.
- **`skilled_match(a, b, rng)`** — each agent capture is a Bernoulli trial whose probability comes from `score_agent_probability(skill_a, skill_b) = 1 / (1 + exp(-(skill_a - skill_b)))` — a logistic (Bradley-Terry) function of the skill gap.
- **`random_match(a, b, rng)`** — each capture is a fair coin flip; skill is completely ignored. This is the null baseline for comparing algorithms.
- **`_tie_break_bonus`** — internal helper. When two games produce tied agent totals, awards one bonus agent. `skilled_match` weights this by skill; `random_match` flips a fair coin.

In [11]:
from tournament.generators import make_players, skilled_match, random_match, score_agent_probability

rng = Random(42)
players = make_players(8, rng, skill_sd=1.0)
pd.DataFrame(
    [(p.pid, p.name, round(p.skill, 3)) for p in players],
    columns=["pid", "name", "skill"]
)

,pid,name,skill
0,0,P000,-0.144
1,1,P001,-0.173
2,2,P002,-0.111
3,3,P003,0.702
4,4,P004,-0.128
5,5,P005,-1.497
6,6,P006,0.332
7,7,P007,-0.267


In [16]:
strong = max(players, key=lambda p: p.skill)
weak   = min(players, key=lambda p: p.skill)
print(f"Strong: {strong.name}  skill={strong.skill:.2f}")
print(f"Weak:   {weak.name}    skill={weak.skill:.2f}")
print(f"Logistic win-prob for strong: {score_agent_probability(strong.skill, weak.skill):.2f}")
print()

n_trials = 200
wins_skilled  = sum(1 for i in range(n_trials) if skilled_match(strong, weak, Random(i)).winner == strong.pid)
wins_random   = sum(1 for i in range(n_trials) if random_match(strong, weak, Random(i)).winner == strong.pid)
draws_skilled = sum(1 for i in range(n_trials) if skilled_match(strong, weak, Random(i)).is_draw)
draws_random  = sum(1 for i in range(n_trials) if random_match(strong, weak, Random(i)).is_draw)

print(f"skilled_match : strong wins {wins_skilled}/{n_trials},  draws {draws_skilled}/{n_trials}")
print(f"random_match  : strong wins {wins_random}/{n_trials},  draws {draws_random}/{n_trials}")

Strong: P003  skill=0.70
Weak:   P005    skill=-1.50
Logistic win-prob for strong: 0.90

skilled_match : strong wins 197/200,  draws 3/200
random_match  : strong wins 50/200,  draws 99/200


## 3. `standings.py` — Records and standings

Computes per-player `Record` objects from a `Tournament`. These are the only input pairing algorithms ever receive.

- **`Record`** accumulates `wins`, `losses`, `agents_for`, `agents_against`, `bonus_agents_won`, and the full sequence of `(agents_for, agents_against)` pairs per match.
- **`compute_records(tournament, through_round=None)`** → `dict[int, Record]`. Pass `through_round=k` to snapshot standings as of round *k*.
- **`group_by_record(records)`** → `dict[str, list[Record]]`. Groups players by exact win-loss record ("3-1", "2-2", etc.), ordered best group first. **Within each group order is unspecified** — call `sort_groups` next.
- **`sort_groups(groups, metric=None)`** → `dict[str, list[Record]]`. Sorts players *within* each group by `make_rank_key(metric)` (best first). Kept as a separate step so callers can choose when (and whether) to sort.
- **`make_rank_key(metric=None)`** → sort-key callable. Returns `(wins, metric(agent_seq))`. `metric` defaults to `agent_differential`. Pass any `(list[(int,int)]) → float` callable to test alternative tiebreakers.

In [17]:
from tournament.engine import run_tournament
from tournament.pairing import get as get_pairing
from tournament.standings import compute_records, group_by_record, sort_groups, make_rank_key, agent_differential

rng = Random(42)
players = make_players(8, rng)
tour = run_tournament(players, n_rounds=4, pairing=get_pairing("adjacent"), rng=rng)

records = compute_records(tour)
skill = {p.pid: p.skill for p in players}

pd.DataFrame([{
    "pid":              r.pid,
    "wins":             r.wins,
    "losses":           r.losses,
    "agents_for":       r.agents_for,
    "agents_against":   r.agents_against,
    "agent_diff":       r.agent_diff,
    "bonus_agents_won": r.bonus_agents_won,
    "true_skill":       round(skill[r.pid], 3),
} for r in records.values()]).sort_values("wins", ascending=False)

,pid,wins,losses,agents_for,agents_against,agent_diff,bonus_agents_won,true_skill
3,3,6,2,20,12,8,0,0.702
0,0,5,3,19,11,8,0,-0.144
6,6,5,3,17,16,1,0,0.332
4,4,5,3,18,14,4,0,-0.128
1,1,4,4,13,14,-1,1,-0.173
7,7,3,5,15,20,-5,1,-0.267
2,2,2,6,14,19,-5,1,-0.111
5,5,2,6,11,21,-10,0,-1.497


In [18]:
# Step 1: group_by_record — pure grouping, no ordering within groups
groups = group_by_record(records)
print("Groups (key: win-loss record, value: list of Records):")
for label, recs in groups.items():
    print(f"  {label}: pids={[r.pid for r in recs]}")

print()

# Step 2: sort_groups — rank within each group by (wins, agent_differential)
sorted_groups = sort_groups(groups)
print("After sort_groups — within each group, best-ranked first:")
for label, recs in sorted_groups.items():
    print(f"  {label}: pids={[r.pid for r in recs]}")

# snapshot after round 2 only
records_r2 = compute_records(tour, through_round=2)
print("\nStandings through round 2 only:", {pid: r.wins for pid, r in records_r2.items()})

Groups (key: win-loss record, value: list of Records):
  6-2: pids=[3]
  5-3: pids=[0, 4, 6]
  4-4: pids=[1]
  3-5: pids=[7]
  2-6: pids=[2, 5]

After sort_groups — within each group, best-ranked first:
  6-2: pids=[3]
  5-3: pids=[0, 4, 6]
  4-4: pids=[1]
  3-5: pids=[7]
  2-6: pids=[2, 5]

Standings through round 2 only: {0: 4, 1: 2, 2: 1, 3: 2, 4: 3, 5: 0, 6: 2, 7: 2}


In [19]:
# Default rank key: (wins, agent_differential)
default_key = make_rank_key()

# Custom metric: total agents captured (ignores opponent's side)
agents_for_only = lambda seq: float(sum(f for f, _ in seq))
custom_key = make_rank_key(metric=agents_for_only)

print("Default ranking: pid → (wins, agent_diff)")
for r in sorted(records.values(), key=default_key, reverse=True):
    print(f"  pid={r.pid}  key={default_key(r)}  agent_diff={r.agent_diff}")

print()
print("Custom ranking: pid → (wins, total_agents_for)")
for r in sorted(records.values(), key=custom_key, reverse=True):
    print(f"  pid={r.pid}  key={custom_key(r)}  agents_for={r.agents_for}")

Default ranking: pid → (wins, agent_diff)
  pid=0  key=8  agent_diff=8
  pid=3  key=8  agent_diff=8
  pid=4  key=4  agent_diff=4
  pid=6  key=1  agent_diff=1
  pid=1  key=-1  agent_diff=-1
  pid=2  key=-5  agent_diff=-5
  pid=7  key=-5  agent_diff=-5
  pid=5  key=-10  agent_diff=-10

Custom ranking: pid → (wins, total_agents_for)
  pid=3  key=20.0  agents_for=20
  pid=0  key=19.0  agents_for=19
  pid=4  key=18.0  agents_for=18
  pid=6  key=17.0  agents_for=17
  pid=7  key=15.0  agents_for=15
  pid=2  key=14.0  agents_for=14
  pid=1  key=13.0  agents_for=13
  pid=5  key=11.0  agents_for=11


## 4. `pairing.py` — Pairing algorithm families

A **pairing function** has the signature `(records, rng, ctx) → list[(a, b)]`. All functions in `REGISTRY` share this signature and can be swapped freely — including between rounds.

**Within-group strategies** (all applied after grouping + sorting by record):
- **`adjacent`** — pair neighbours in rank order: 1v2, 3v4, … (most competitive within a bracket)
- **`fold`** — top half vs bottom half: rank 1 plays rank N/2+1, rank 2 plays rank N/2+2, …
- **`strong_weak`** — best vs worst, second-best vs second-worst, etc.
- **`random_within_record`** — shuffle within each record group (controlled randomness)
- **`random`** — ignore records entirely; shuffle the whole field (null baseline)

**Supporting objects:**
- **`PairingContext(past_opponents, round_number)`** — carries the rematch avoidance table and round number.
- **`_avoid_rematches`** — greedy best-effort swap; not guaranteed for pathological fields (which is itself a measurable phenomenon).
- **`make_record_group_pairing(strategy, metric=None)`** — factory. Builds a `PairingFunction` for any named strategy; threads `metric` to `sort_groups`.
- **`REGISTRY`** / **`get(name)`** — look up a pre-built function by name string.

In [21]:
from tournament.pairing import REGISTRY, get as get_pairing, make_record_group_pairing, PairingContext

print("Available pairing algorithms:")
for name, fn in REGISTRY.items():
    print(f"  {name!r:25s}  (fn name: {fn.__name__})")

Available pairing algorithms:
  'adjacent'                 (fn name: record_group__adjacent)
  'fold'                     (fn name: record_group__fold)
  'strong_weak'              (fn name: record_group__strong_weak)
  'random_within_record'     (fn name: record_group__random_within_record)
  'random'                   (fn name: random_pairing)


In [22]:
# Manually call a pairing function — exactly what engine.py does each round.
rng2 = Random(7)
players2 = make_players(8, rng2)
tour_partial = run_tournament(players2, n_rounds=2, pairing=get_pairing("adjacent"), rng=rng2)
records2 = compute_records(tour_partial)

ctx = PairingContext(
    past_opponents={p.pid: tour_partial.past_opponents(p.pid) for p in players2},
    round_number=3,
)

for name in ["adjacent", "fold", "strong_weak"]:
    pairs = get_pairing(name)(records2, rng2, ctx)
    print(f"{name:20s}: {pairs}")

adjacent            : [(6, 1), (5, 7), (2, 0), (3, 4)]
fold                : [(6, 1), (5, 7), (2, 0), (3, 4)]
strong_weak         : [(6, 1), (5, 2), (7, 0), (4, 3)]


## 5. `engine.py` — The round loop

`run_tournament` orchestrates the full simulation:

1. For round *n*, calls `compute_records(tournament)` on rounds 1..n−1.
2. Builds a `PairingContext` (past opponents + round number).
3. Calls `_pairing_for_round(spec, n)` to resolve which function to use.
4. Calls `fn(records, rng, ctx)` → list of `(a, b)` pairs.
5. Plays each pair with `match_model(Player, Player, rng)` → `Match`.
6. Appends the new `Round` and repeats.

The `pairing` argument is flexible:
- **Single function** — reused every round.
- **List** — round 1 uses index 0, round 2 index 1, … clamped at last entry.
- **Dict `{round_number: fn}`** — falls back to the latest key ≤ current round.

`match_model` defaults to `skilled_match`; pass `random_match` to run skill-blind.

In [23]:
from tournament.generators import random_match

rng3 = Random(99)
players3 = make_players(8, rng3)

# --- Single function reused every round ---
tour_adj = run_tournament(players3, n_rounds=4, pairing=get_pairing("adjacent"), rng=rng3)
print(f"adjacent:  {len(tour_adj.rounds)} rounds, {sum(len(r.matches) for r in tour_adj.rounds)} matches")

# --- List schedule: random round 1 (no info yet), then fold ---
schedule = [get_pairing("random"), get_pairing("fold"), get_pairing("fold"), get_pairing("fold")]
tour_sched = run_tournament(players3, n_rounds=4, pairing=schedule, rng=Random(99))
print(f"schedule:  rounds used {[fn.__name__ for fn in schedule]}")

# --- Dict spec: explicit per-round functions ---
spec = {1: get_pairing("random"), 2: get_pairing("adjacent")}  # round 3+ falls back to adjacent
tour_dict = run_tournament(players3, n_rounds=4, pairing=spec, rng=Random(99))
print(f"dict spec: {len(tour_dict.rounds)} rounds")

# --- Skill-blind: random_match ignores Player.skill ---
tour_blind = run_tournament(players3, n_rounds=4, pairing=get_pairing("adjacent"),
                            rng=Random(99), match_model=random_match)
print(f"skill-blind run: {len(tour_blind.rounds)} rounds")

adjacent:  4 rounds, 16 matches
schedule:  rounds used ['random_pairing', 'record_group__fold', 'record_group__fold', 'record_group__fold']
dict spec: 4 rounds
skill-blind run: 4 rounds


## 6. `metrics.py` — Quality metrics

Evaluates a finished tournament against the latent `skill` values (which algorithms never saw).

- **`mean_skill_gap(tournament)`** — average absolute skill difference between paired players across all rounds. *Lower = better matched.*
- **`standings_skill_correlation(tournament)`** — Pearson correlation between final standings rank and true skill rank. *+1 = standings perfectly recover skill order; 0 = no relationship.*
- **`rematch_count(tournament)`** — how many times any pair of players met more than once.
- **`summary(tournament)`** — returns all three plus `rounds` and `players` counts as a dict.

> Note: the choice of headline evaluation metric is an open design decision (see `docs/decisions.md` D4 and the `# Windsurf:` comment in the file).

In [24]:
from tournament.metrics import mean_skill_gap, standings_skill_correlation, rematch_count, summary
from statistics import fmean

rng4 = Random(42)
players4 = make_players(16, rng4)
tour4 = run_tournament(players4, n_rounds=5, pairing=get_pairing("adjacent"), rng=rng4)

print("Single-tournament metrics (adjacent, 16 players, 5 rounds):")
print(f"  mean_skill_gap:              {mean_skill_gap(tour4):.4f}  (lower = better matched)")
print(f"  standings_skill_correlation: {standings_skill_correlation(tour4):.4f}  (higher = standings reflect skill)")
print(f"  rematch_count:               {rematch_count(tour4)}")
print()

# Average over 100 trials per strategy
def avg_metrics(name, trials=100, n_players=16, n_rounds=5):
    rows = [
        summary(run_tournament(make_players(n_players, Random(t)), n_rounds, get_pairing(name), Random(t + 1)))
        for t in range(trials)
    ]
    return {k: round(fmean(r[k] for r in rows), 4) for k in rows[0]}

comparison = pd.DataFrame({name: avg_metrics(name) for name in REGISTRY}).T
comparison[["mean_skill_gap", "standings_skill_correlation", "rematch_count"]]

Single-tournament metrics (adjacent, 16 players, 5 rounds):
  mean_skill_gap:              0.5413  (lower = better matched)
  standings_skill_correlation: 0.8412  (higher = standings reflect skill)
  rematch_count:               5



,mean_skill_gap,standings_skill_correlation,rematch_count
adjacent,0.8545,0.8627,2.27
fold,0.8566,0.8750,2.36
strong_weak,0.8629,0.8507,2.75
random_within_record,0.8773,0.8471,2.66
random,1.1240,0.8439,2.60


## 7. `io.py` — CSV persistence

Saves and restores tournament data in flat, tool-agnostic CSV files (Excel / pandas / R friendly).

- **`write_matches(tournament, path)`** — one row per match; stores both games and `bonus_agents_a/b` so the full match round-trips exactly.
- **`write_standings(tournament, path, every_round=True)`** — one row per player per round snapshot. Useful for studying how rankings evolve over time.
- **`write_players(tournament, path)`** — saves the player pool including latent skill (ground truth).
- **`read_matches(path, players)`** — reconstructs a `Tournament` from a matches CSV, restoring bonus agents correctly.

In [ ]:
import tempfile
from pathlib import Path
from tournament.io import write_matches, write_standings, write_players, read_matches

rng5 = Random(42)
players5 = make_players(8, rng5)
tour5 = run_tournament(players5, n_rounds=3, pairing=get_pairing("adjacent"), rng=rng5)

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir)
    write_matches(tour5, p / "matches.csv")
    write_standings(tour5, p / "standings.csv")
    write_players(tour5, p / "players.csv")

    # Round-trip: restore from CSV and verify bonus agents survived
    tour_restored = read_matches(p / "matches.csv", players5)
    m_orig = tour5.rounds[0].matches[0]
    m_rest = tour_restored.rounds[0].matches[0]
    assert m_orig.total_agents_a == m_rest.total_agents_a, "total_agents_a round-trip failed"
    assert m_orig.total_agents_b == m_rest.total_agents_b, "total_agents_b round-trip failed"
    print(f"Round-trip OK — {len(tour_restored.rounds)} rounds restored")

    matches_df  = pd.read_csv(p / "matches.csv")
    standings_df = pd.read_csv(p / "standings.csv")

print("\nmatches.csv columns:", list(matches_df.columns))
matches_df.head(6)